In [10]:
import threading
from datetime import datetime
import queue
import sys
import os


In [15]:
class logger(threading.Thread):
    """
    后台日志写入线程。
    负责从队列中取出日志消息并写入文件/控制台。
    """
    def __init__(self, name="AVC", print_to_console=True):
        super().__init__(daemon=True)       # 设置为守护线程，主程序退出时自动结束
        self.log_file = os.path.join("../logs", f"log_{datetime.now().strftime('%Y%m%d_%H%M')}.txt")

        self.log_queue = queue.Queue(-1)
        self.running = True
        self.node_name = name

        self.print_to_console = print_to_console  # 是否同时输出到控制台

        self.info("Logger thread initialized.")

    def run(self):
        """
        线程主循环：不断从队列取消息并处理。
        """
        while self.running or not self.log_queue.empty():
            try:
                # 设置 timeout 防止死锁，允许每隔一段时间检查 running 状态
                record = self.log_queue.get(timeout=0.5)
                with open(self.log_file, 'a', encoding='utf-8') as f:
                    f.write(record + '\n')
                if self.print_to_console:
                    print(record)
            except queue.Empty:
                continue
            except Exception as e:
                print(f"Logging thread error: {e}", file=sys.stderr)

    def info(self, message):
        """
        记录信息级别日志。
        """
        timestamp = datetime.now().strftime('%H:%M:%S')
        log_record = f"[INFO][{self.node_name}][{timestamp}] {message}"
        self.log_queue.put(log_record)

    def warning(self, message):
        """
        记录警告级别日志。
        """
        timestamp = datetime.now().strftime('%H:%M:%S')
        log_record = f"[WARNING][{self.node_name}][{timestamp}] {message}"
        self.log_queue.put(log_record)

    def error(self, message):
        """
        记录错误级别日志。
        """
        timestamp = datetime.now().strftime('%H:%M:%S')
        log_record = f"[ERROR][{self.node_name}][{timestamp}] {message}"
        self.log_queue.put(log_record)

    def stop(self):
        """
        停止日志线程，等待其处理完队列中的消息。
        """
        self.running = False
        self.join()  # 等待线程结束
        timestamp = datetime.now().strftime('%H:%M:%S')
        exit_msg = f"[INFO][{self.node_name}][{timestamp}]Logger thread stopped. Log file: {self.log_file}"
        print(exit_msg)

In [16]:
logger = logger(print_to_console=True)
logger.start()
logger.info("Logger initialized and running.")
logger.warning("This is a warning message.")
logger.error("This is an error message.")
logger.stop()

[INFO][AVC][17:53:19] Logger thread initialized.
[INFO][AVC][17:53:19] Logger initialized and running.
[WARNING][AVC][17:53:19] This is a warning message.
[ERROR][AVC][17:53:19] This is an error message.
[INFO][AVC][17:53:19]Logger thread stopped. Log file: ../logs/log_20260317_1753.txt
